# 03 Feature extraction

Extracts frozen image features, and text features of the KG disease descriptions, from three backbones:

| Key | Model | Text encoder | Role |
|---|---|---|---|
| `monet` | MONET: CLIP ViT-L/14 trained on dermatology image-text pairs | yes | main backbone |
| `clip_l14` | OpenAI CLIP ViT-L/14 | yes | general-domain vision-language |
| `dinov2_l` | DINOv2 ViT-L/14 | no | vision-only: class semantics come only from the KG |

Weights are loaded from local folders under `models/`. Step 1 downloads missing weights through the mirror (with retries), or prints the files to fetch by browser.

Everything downstream trains on the cached features, so this notebook runs once.

**Outputs** (`data_v2/features/<backbone>/`): `image.npy` (rows follow `meta.csv`), `text_<view>.npy` (rows follow `disease_id`), `info.json`; plus `data_v2/features/sanity_baselines.csv`.

`pip install torch torchvision transformers huggingface_hub pandas numpy pillow tqdm`

In [1]:
# ============ Config ============
# environment variables must be set before transformers / huggingface_hub are imported
import os
from pathlib import Path

HF_ENDPOINT = 'https://hf-mirror.com'      # mainland China mirror; None for huggingface.co
if HF_ENDPOINT:
    os.environ['HF_ENDPOINT'] = HF_ENDPOINT
os.environ['HF_HUB_DISABLE_XET'] = '1'      # Xet storage often fails through mirrors
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

DATA      = Path('data_v2')
OUT       = DATA / 'features'
MODEL_DIR = Path('models')

# repo: Hugging Face id; local: folder under MODEL_DIR that the weights are loaded from
BACKBONES = {
    'monet':    dict(repo='suinleelab/monet',              local='monet',                  kind='clip'),
    'clip_l14': dict(repo='openai/clip-vit-large-patch14', local='clip-vit-large-patch14', kind='clip'),
    'dinov2_l': dict(repo='facebook/dinov2-large',         local='dinov2-large',           kind='dino'),
}
RUN        = ['monet', 'clip_l14', 'dinov2_l']
DOWNLOAD   = True     # download missing weights into MODEL_DIR; False: local files only
BATCH      = 64       # RTX 5070 (12 GB): 64-128; CPU: 16-32
N_THREADS  = 8        # image decoding threads (thread-based, safe on Windows)
HFLIP_TTA  = False    # average features of the image and its horizontal flip
FORCE      = False    # True: re-extract even if features exist
TEXT_VIEWS = ['name_text', 'attr_text', 'full_text', 'full_sent']
KNN_K      = 10

OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# environment check
import platform
import torch

print('python', platform.python_version(), '| torch', torch.__version__, '| CUDA build', torch.version.cuda)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} | capability {torch.cuda.get_device_capability(0)} | {p.total_memory / 1e9:.1f} GB')
else:
    print('no GPU visible: running on CPU')
    if torch.version.cuda is None:
        print('this torch build is CPU-only. For an RTX 50-series GPU install a CUDA 12.8+ build:')
        print('  pip uninstall torch torchvision -y')
        print('  pip install torch torchvision -f https://mirrors.aliyun.com/pytorch-wheels/cu128/')
## 1. Model weights

python 3.11.18 | torch 2.11.0+cu128 | CUDA build 12.8
GPU: NVIDIA GeForce RTX 5070 Laptop GPU | capability (12, 0) | 8.5 GB


In [3]:
from huggingface_hub import snapshot_download

PATTERNS = ['*.json', '*.txt', '*.safetensors']     # skip .bin, TensorFlow and Flax copies

def weights_ready(path):
    return (path / 'config.json').exists() and any(path.glob('*.safetensors'))

for name in RUN:
    spec = BACKBONES[name]
    path = MODEL_DIR / spec['local']
    if weights_ready(path):
        print(f'[{name}] ready: {path}')
        continue
    if DOWNLOAD:
        for attempt in range(5):
            try:
                snapshot_download(spec['repo'], local_dir=str(path), allow_patterns=PATTERNS)
                break
            except Exception as e:
                print(f'[{name}] attempt {attempt + 1} failed: {type(e).__name__}: {str(e)[:160]}')
    if weights_ready(path):
        print(f'[{name}] downloaded: {path}')
    else:
        base = (HF_ENDPOINT or 'https://huggingface.co').rstrip('/')
        print(f'[{name}] NOT READY. Download by browser into {path.resolve()}:')
        print(f'    {base}/{spec["repo"]}/tree/main')
        print('    every *.json and *.txt file, plus model.safetensors')

c:\Users\78479\anaconda3\envs\skin311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[monet] ready: models\monet
[clip_l14] ready: models\clip-vit-large-patch14
[dinov2_l] ready: models\dinov2-large


## 2. Data

In [4]:
import json, time, gc, re
import numpy as np
import pandas as pd
import torch.nn.functional as F
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer

DEVICE = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available() else 'cpu')
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
if DEVICE == 'cpu':
    torch.set_num_threads(os.cpu_count())

meta = pd.read_csv(DATA / 'meta.csv')
text = pd.read_csv(DATA / 'kg' / 'disease_text.csv').sort_values('disease_id').reset_index(drop=True)
img_paths = meta['cache_path'].where(meta['cache_path'].notna(), meta['path']).tolist()
n_cls = len(text)
assert (text['disease_id'].to_numpy() == np.arange(n_cls)).all() and meta['disease_id'].max() < n_cls, \
    'meta.csv and disease_text.csv disagree: rerun 01 and 02'
print(f'device: {DEVICE} | {len(meta)} images | {n_cls} diseases')

W1121 03:05:52.219000 46008 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


device: cuda | 16321 images | 114 diseases


## 3. Model helpers

In [5]:
def load_backbone(spec):
    path = MODEL_DIR / spec['local']
    assert weights_ready(path), f'weights missing in {path}: see step 1'
    model = AutoModel.from_pretrained(path).to(DEVICE, DTYPE).eval()
    proc = AutoImageProcessor.from_pretrained(path)
    tok = AutoTokenizer.from_pretrained(path) if spec['kind'] == 'clip' else None
    return model, proc, tok

def as_tensor(out):
    # transformers 4.x returns a tensor, 5.x may return a ModelOutput
    if torch.is_tensor(out):
        return out
    return out.pooler_output if getattr(out, 'pooler_output', None) is not None else out[0]

@torch.no_grad()
def encode_images(model, kind, px):
    px = px.to(DEVICE, DTYPE)
    if kind == 'clip':
        return as_tensor(model.get_image_features(pixel_values=px))
    return model(pixel_values=px).pooler_output       # DINOv2: layer-normed CLS token

@torch.no_grad()
def encode_texts(model, tok, texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        t = tok(texts[i:i + bs], padding=True, truncation=True, max_length=77, return_tensors='pt').to(DEVICE)
        out.append(as_tensor(model.get_text_features(**t)).float())
    return torch.cat(out)

def load_image(p):
    with Image.open(p) as im:
        return im.convert('RGB')

## 4. Extract

In [6]:
def extract(name, spec):
    d = OUT / name
    d.mkdir(exist_ok=True)
    if (d / 'image.npy').exists() and not FORCE:
        print(f'[{name}] features exist, skipping (FORCE=True to redo)')
        return
    t0 = time.time()
    model, proc, tok = load_backbone(spec)

    # image features; the next batch is decoded while the current one runs on the GPU
    feats = []
    batches = [img_paths[i:i + BATCH] for i in range(0, len(img_paths), BATCH)]
    with ThreadPoolExecutor(N_THREADS) as ex:
        load_batch = lambda b: list(ex.map(load_image, b))
        nxt = ex.submit(load_batch, batches[0])
        for k in tqdm(range(len(batches)), desc=name):
            ims = nxt.result()
            if k + 1 < len(batches):
                nxt = ex.submit(load_batch, batches[k + 1])
            px = proc(images=ims, return_tensors='pt')['pixel_values']
            f = encode_images(model, spec['kind'], px).float()
            if HFLIP_TTA:
                f = (f + encode_images(model, spec['kind'], px.flip(-1)).float()) / 2
            feats.append(f.cpu())
    feats = torch.cat(feats).numpy()
    np.save(d / 'image.npy', feats.astype(np.float16))

    # text features of the KG descriptions (vision-language backbones only)
    if tok is not None:
        for v in TEXT_VIEWS:
            if v == 'full_sent':
                # the CLIP text encoder truncates at 77 tokens: encode sentence by sentence and average
                embs = []
                for s in text['full_text']:
                    sents = [x for x in re.split(r'(?<=[.!?])\s+', s) if x]
                    embs.append(F.normalize(encode_texts(model, tok, sents), dim=-1).mean(0))
                T = torch.stack(embs)
            else:
                T = encode_texts(model, tok, text[v].tolist())
            np.save(d / f'text_{v}.npy', T.cpu().numpy().astype(np.float32))

    info = dict(backbone=name, repo=spec['repo'], kind=spec['kind'], n_images=int(len(feats)),
                dim=int(feats.shape[1]), hflip_tta=HFLIP_TTA, device=DEVICE,
                minutes=round((time.time() - t0) / 60, 1), row_order='meta.csv')
    json.dump(info, open(d / 'info.json', 'w'), indent=2)
    print(f'[{name}] {feats.shape} in {info["minutes"]} min')
    del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

for name in RUN:
    if weights_ready(MODEL_DIR / BACKBONES[name]['local']):
        extract(name, BACKBONES[name])
    else:
        print(f'[{name}] skipped: weights not ready')

monet: 100%|██████████| 256/256 [04:06<00:00,  1.04it/s]


[monet] (16321, 768) in 4.2 min


clip_l14: 100%|██████████| 256/256 [02:43<00:00,  1.57it/s]


[clip_l14] (16321, 768) in 2.8 min


dinov2_l: 100%|██████████| 256/256 [02:00<00:00,  2.13it/s]


[dinov2_l] (16321, 1024) in 2.0 min


## 5. Sanity checks

- **Text zero-shot** (vision-language backbones): cosine similarity between image features and disease text features, on the closed-set test split over all diseases, and on the unseen diseases only (ZSL).
- **kNN** (all backbones): cosine kNN from the closed-set training split to its test split, a quick measure of how disease-discriminative the frozen features are.

These are reference baselines, not the proposed method.

In [7]:
y  = meta['disease_id'].to_numpy()
te = (meta['split_lt'] == 'test').to_numpy()
tr = (meta['split_lt'] == 'train').to_numpy()
zs = (meta['split_zsl'] == 'test_unseen').to_numpy()
unseen_ids = np.sort(meta.loc[meta['unseen'], 'disease_id'].unique())

def metrics(scores, yt, classes=None):
    if classes is not None:                     # restrict prediction to a class subset
        scores = scores[:, classes]
        yt = np.searchsorted(classes, yt)
    top = np.argsort(-scores, 1)[:, :5]
    acc1 = (top[:, 0] == yt).mean()
    acc5 = (top == yt[:, None]).any(1).mean()
    bacc = np.mean([(top[yt == c, 0] == c).mean() for c in np.unique(yt)])
    return acc1, acc5, bacc

rows = []
for name in RUN:
    d = OUT / name
    if not (d / 'image.npy').exists():
        continue
    X = F.normalize(torch.from_numpy(np.load(d / 'image.npy').astype(np.float32)), dim=-1)

    if BACKBONES[name]['kind'] == 'clip':
        for v in TEXT_VIEWS:
            T = F.normalize(torch.from_numpy(np.load(d / f'text_{v}.npy')), dim=-1)
            S = (X @ T.T).numpy()
            a1, a5, ba = metrics(S[te], y[te])
            z1, z5, zb = metrics(S[zs], y[zs], classes=unseen_ids)
            rows.append(dict(backbone=name, method=f'zero-shot [{v}]', top1=a1, top5=a5, bacc=ba,
                             zsl_top1=z1, zsl_bacc=zb))

    sim = X[te] @ X[tr].T
    val, idx = sim.topk(KNN_K, dim=1)
    votes = np.zeros((te.sum(), n_cls))
    np.add.at(votes, (np.repeat(np.arange(te.sum()), KNN_K), y[tr][idx.numpy()].ravel()), val.numpy().ravel())
    a1, a5, ba = metrics(votes, y[te])
    rows.append(dict(backbone=name, method=f'kNN (k={KNN_K})', top1=a1, top5=a5, bacc=ba))

res = pd.DataFrame(rows)
res.to_csv(OUT / 'sanity_baselines.csv', index=False)
print(f'chance top-1: {1 / n_cls:.3f} (all diseases) | {1 / len(unseen_ids):.3f} (unseen only)')
res.round(3)

C:\Users\78479\AppData\Local\Temp\ipykernel_46008\984015277.py:33: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  sim = X[te] @ X[tr].T


chance top-1: 0.009 (all diseases) | 0.043 (unseen only)


,backbone,method,top1,top5,bacc,zsl_top1,zsl_bacc
0,monet,zero-shot [name_text],0.165,0.417,0.192,0.333,0.388
1,monet,zero-shot [attr_text],0.168,0.407,0.195,0.333,0.426
2,monet,zero-shot [full_text],0.171,0.421,0.200,0.351,0.445
3,monet,zero-shot [full_sent],0.150,0.391,0.184,0.316,0.398
4,monet,kNN (k=10),0.478,0.742,0.436,NaN,NaN
5,clip_l14,zero-shot [name_text],0.091,0.250,0.096,0.206,0.217
6,clip_l14,zero-shot [attr_text],0.077,0.233,0.099,0.196,0.254
7,clip_l14,zero-shot [full_text],0.073,0.232,0.103,0.163,0.243
8,clip_l14,zero-shot [full_sent],0.046,0.180,0.061,0.145,0.190
9,clip_l14,kNN (k=10),0.471,0.755,0.426,NaN,NaN
